In [ ]:
# ============================================================
# รันก่อนเลย (สำหรับ Google Colab)
# ============================================================
# !pip install scikit-learn pandas numpy matplotlib seaborn xgboost -q
print('✅ ติดตั้งเสร็จแล้ว รัน cell ถัดไปได้เลย')

# 🧪 EXAM CHEAT SHEET – 14 มีนาคม
**หัวข้อ:** K-Means | Linear/Non-linear Regression | Logistic Regression | KNN | Decision Tree | Ensemble

---
## ⚡ QUICK JUMP
| Topic | Section |
|-------|---------|
| K-Means Clustering | Cell 2 |
| Linear Regression | Cell 3 |
| Non-linear Regression | Cell 4 |
| Logistic Regression | Cell 5 |
| KNN | Cell 6 |
| Decision Tree | Cell 7 |
| Random Forest | Cell 8 |
| Ensemble (AdaBoost/GBM/XGB) | Cell 9 |
| Evaluation (ทุกแบบ) | Cell 10 |
| Data Prep Template | Cell 11 |

In [ ]:
# ============================================================
# IMPORTS – รันก่อนเสมอ
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)
import warnings; warnings.filterwarnings('ignore')
print('Ready!')

---
# 🔄 CRISP-DM WORKFLOW

| Phase | หัวข้อใน Cheatsheet |
|-------|---------------------|
| **1. Business Understanding** | กำหนดเป้าหมาย / ประเภท task (Classification / Regression / Clustering) |
| **2. Data Understanding** | EDA & Visualization → Section 0.5 ↓ |
| **3. Data Preparation** | Data Prep Template → Section 10 / Encoding → Section 10.5 |
| **4. Modeling** | KMeans / Linear / Logistic / KNN / DT / RF / Ensemble → Section 1–8 |
| **5. Evaluation** | Metrics, CV, GridSearch → Section 9, 9.5 |
| **6. Deployment** | Export model: `joblib.dump(model, 'model.pkl')` |


---
# 0.5 EDA & VISUALIZATION 📊
**Phase 2: Data Understanding**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

# ---- โหลดข้อมูล (ใช้ breast_cancer เป็นตัวอย่าง) ----
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df['target_name'] = df['target'].map({0: 'malignant', 1: 'benign'})

# ============================================================
# Phase 2.1 – Data Overview
# ============================================================
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes.value_counts())
print("\nMissing Values:\n", df.isnull().sum().sum(), "total missing")
print("\nClass Distribution:\n", df['target_name'].value_counts())
df.describe().round(2)


In [ ]:
# ============================================================
# Phase 2.2 – Distribution Plots
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

cols = data.feature_names[:6]   # แสดง 6 features แรก

# ---- Histogram (distribution) ----
for i, col in enumerate(cols[:3]):
    axes[0, i].hist(df[df['target']==1][col], bins=20, alpha=0.6, color='steelblue', label='benign')
    axes[0, i].hist(df[df['target']==0][col], bins=20, alpha=0.6, color='tomato', label='malignant')
    axes[0, i].set_title(f'Histogram: {col}', fontsize=9)
    axes[0, i].legend(fontsize=8)

# ---- Boxplot (outliers + spread per class) ----
for i, col in enumerate(cols[:3]):
    df.boxplot(column=col, by='target_name', ax=axes[1, i])
    axes[1, i].set_title(f'Boxplot: {col}', fontsize=9)
    axes[1, i].set_xlabel('')

plt.suptitle('Distribution & Boxplots', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Phase 2.3 – Correlation Heatmap 🔥
# ============================================================
# เลือก features ที่ corr สูงสุดกับ target
corr_matrix = df.drop('target_name', axis=1).corr()

# ---- Full heatmap ----
plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))   # ซ่อนครึ่งบน (ซ้ำ)
sns.heatmap(
    corr_matrix, mask=mask,
    cmap='coolwarm', center=0,
    annot=False, linewidths=0.3,
    vmin=-1, vmax=1
)
plt.title('Correlation Heatmap (lower triangle)', fontsize=13)
plt.tight_layout()
plt.show()

# ---- Top corr with target ----
top_corr = corr_matrix['target'].abs().sort_values(ascending=False).head(11)[1:]
print("Top 10 features correlated with target:")
print(top_corr.round(3))

# ---- Focused heatmap (top features only) ----
top_feats = top_corr.index.tolist() + ['target']
plt.figure(figsize=(10, 8))
sns.heatmap(
    df[top_feats].corr(),
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5
)
plt.title('Correlation Heatmap – Top Features', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Phase 2.4 – Countplot, Pairplot, Scatter
# ============================================================

# ---- Countplot (class distribution) ----
plt.figure(figsize=(5, 3))
sns.countplot(x='target_name', data=df, palette=['tomato', 'steelblue'])
plt.title('Class Distribution (Countplot)')
plt.xlabel('Class'); plt.ylabel('Count')
for p in plt.gca().patches:
    plt.gca().annotate(f'{int(p.get_height())}', (p.get_x()+0.3, p.get_height()+2))
plt.tight_layout()
plt.show()

# ---- Pairplot (top 4 features) ----
pair_cols = top_corr.index[:4].tolist() + ['target_name']
sns.pairplot(df[pair_cols], hue='target_name',
             palette={'benign': 'steelblue', 'malignant': 'tomato'},
             plot_kws={'alpha': 0.5, 's': 20}, diag_kind='kde')
plt.suptitle('Pairplot – Top 4 Features', y=1.01, fontsize=12)
plt.show()

# ---- Scatter (2 features) ----
plt.figure(figsize=(7, 5))
for label, color in zip([0, 1], ['tomato', 'steelblue']):
    mask = df['target'] == label
    plt.scatter(
        df.loc[mask, top_corr.index[0]],
        df.loc[mask, top_corr.index[1]],
        label=data.target_names[label], alpha=0.6, s=25, c=color
    )
plt.xlabel(top_corr.index[0]); plt.ylabel(top_corr.index[1])
plt.title('Scatter Plot – Top 2 Features'); plt.legend()
plt.tight_layout(); plt.show()


---
# 1. K-MEANS CLUSTERING 🔵

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ---- โหลดข้อมูล (เปลี่ยนตรงนี้) ----
# df = pd.read_csv('file.csv')
# X = df[['col1', 'col2']]          # เลือก features

# ---- ตัวอย่างใช้ iris ----
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
X = iris.data

# ---- Scale ----
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ---- Elbow Method หา k ที่ดี ----
inertias = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7,4))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('k'); plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.grid(True)
plt.show()

# ---- Fit KMeans ----
k = 3   # <-- เปลี่ยนตาม elbow
km = KMeans(n_clusters=k, random_state=42, n_init=10)
km.fit(X_scaled)
labels = km.labels_

# ---- เพิ่ม cluster กลับใน df ----
df_result = iris.data.copy()
df_result['cluster'] = labels

# ---- สรุปแต่ละ cluster ----
print(df_result.groupby('cluster').mean().round(2))

# ---- Scatter plot ----
plt.figure(figsize=(7,5))
for c in range(k):
    mask = labels == c
    plt.scatter(X_scaled[mask, 0], X_scaled[mask, 1], label=f'Cluster {c}', alpha=0.7)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            marker='X', s=200, c='black', label='Centroids')
plt.legend(); plt.title('K-Means Clusters')
plt.show()

# ---- Silhouette Score ----
from sklearn.metrics import silhouette_score
sil = silhouette_score(X_scaled, labels)
print(f'Silhouette Score: {sil:.3f}  (ใกล้ 1 = ดี)')

---
# 2. LINEAR REGRESSION 📈

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# ---- โหลดข้อมูล (เปลี่ยนตรงนี้) ----
# df = pd.read_csv('file.csv')
# X = df.drop('target_col', axis=1)
# y = df['target_col']

# ---- ตัวอย่าง ----
from sklearn.datasets import load_diabetes
data = load_diabetes()
X, y = data.data, data.target

# ---- Split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---- Scale (optional แต่แนะนำ) ----
sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s  = sc.transform(X_test)

# ---- Train ----
model = LinearRegression()
# model = Ridge(alpha=1.0)    # ถ้า overfit
# model = Lasso(alpha=0.1)    # ถ้าต้องการ feature selection
model.fit(X_train_s, y_train)

# ---- Predict ----
y_pred = model.predict(X_test_s)

# ---- Evaluate ----
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)
print(f'MAE:  {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R²:   {r2:.4f}  (1.0 = perfect)')

# ---- Coefficients ----
coef_df = pd.DataFrame({'feature': data.feature_names, 'coef': model.coef_})
print(coef_df.sort_values('coef', ascending=False))

# ---- Actual vs Predicted ----
plt.figure(figsize=(7,5))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual'); plt.ylabel('Predicted')
plt.title(f'Actual vs Predicted  (R²={r2:.3f})')
plt.show()

---
# 3. NON-LINEAR REGRESSION 📈〰️

In [ ]:
# ======================================================
# วิธีที่ 1: Polynomial Features
# ======================================================
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

# ตัวอย่าง data non-linear
np.random.seed(42)
X_nl = np.linspace(-3, 3, 100).reshape(-1,1)
y_nl = 0.5*X_nl.ravel()**3 - X_nl.ravel()**2 + np.random.randn(100)*2

X_tr, X_te, y_tr, y_te = train_test_split(X_nl, y_nl, test_size=0.2, random_state=42)

# Pipeline: PolynomialFeatures → LinearRegression
poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),  # degree=2,3,4...
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
poly_pipeline.fit(X_tr, y_tr)
y_pred_nl = poly_pipeline.predict(X_te)

print(f'Polynomial R²: {r2_score(y_te, y_pred_nl):.4f}')

# Plot
X_plot = np.linspace(-3, 3, 300).reshape(-1,1)
plt.figure(figsize=(8,4))
plt.scatter(X_nl, y_nl, alpha=0.5, s=20, label='data')
plt.plot(X_plot, poly_pipeline.predict(X_plot), 'r-', lw=2, label='poly fit')
plt.legend(); plt.title('Polynomial Regression (degree=3)')
plt.show()

# ======================================================
# วิธีที่ 2: Decision Tree / SVR สำหรับ Non-linear
# ======================================================
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

dtr = DecisionTreeRegressor(max_depth=4, random_state=42)
dtr.fit(X_tr, y_tr)
print(f'DecisionTree R²: {r2_score(y_te, dtr.predict(X_te)):.4f}')

svr = SVR(kernel='rbf', C=100, epsilon=0.5)   # kernel='rbf' คือ non-linear
svr.fit(X_tr, y_tr)
print(f'SVR (rbf) R²: {r2_score(y_te, svr.predict(X_te)):.4f}')

---
# 4. LOGISTIC REGRESSION 🎯

In [ ]:
from sklearn.linear_model import LogisticRegression

# ---- โหลดข้อมูล (เปลี่ยนตรงนี้) ----
# df = pd.read_csv('file.csv')
# X = df.drop('target', axis=1)
# y = df['target']

# ---- Encode categorical (ถ้าจำเป็น) ----
# df = pd.get_dummies(df, drop_first=True)
# หรือ
# le = LabelEncoder()
# df['col'] = le.fit_transform(df['col'])

# ---- ตัวอย่าง ----
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # stratify สำคัญ!
)
sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s  = sc.transform(X_test)

# ---- Train ----
model = LogisticRegression(
    max_iter=1000,          # เพิ่มถ้า convergence warning
    C=1.0,                  # regularization (เล็ก = regularize มาก)
    solver='lbfgs',         # binary/multiclass
)
model.fit(X_train_s, y_train)

# ---- Predict ----
y_pred  = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)  # probability

# ---- Evaluate ----
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=data.target_names))

# ---- Confusion Matrix ----
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=data.target_names, yticklabels=data.target_names)
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

# ---- ROC AUC (binary) ----
from sklearn.metrics import roc_auc_score
print(f'AUC-ROC: {roc_auc_score(y_test, y_proba[:,1]):.4f}')


---
# 5. K-NEAREST NEIGHBORS (KNN) 🔷

In [ ]:
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

# ---- ตัวอย่าง ----
from sklearn.datasets import load_iris
X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# KNN ต้อง Scale เสมอ!
sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s  = sc.transform(X_test)

# ---- หา K ที่ดีที่สุด ----
k_range = range(1, 21)
train_scores, test_scores = [], []
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    train_scores.append(knn.score(X_train_s, y_train))
    test_scores.append(knn.score(X_test_s, y_test))

plt.figure(figsize=(8,4))
plt.plot(k_range, train_scores, 'b-o', label='Train')
plt.plot(k_range, test_scores,  'r-s', label='Test')
plt.xlabel('k'); plt.ylabel('Accuracy')
plt.title('KNN: k vs Accuracy')
plt.legend(); plt.grid(True)
plt.show()

best_k = k_range[np.argmax(test_scores)]
print(f'Best k: {best_k}, Test Accuracy: {max(test_scores):.4f}')

# ---- Train best model ----
knn = KNeighborsClassifier(
    n_neighbors=best_k,
    metric='euclidean',      # 'euclidean','manhattan','minkowski'
    weights='uniform'        # 'uniform' หรือ 'distance'
)
knn.fit(X_train_s, y_train)
y_pred = knn.predict(X_test_s)

print(classification_report(y_test, y_pred))

# ---- KNN Regression (ถ้า task เป็น regression) ----
# knn_reg = KNeighborsRegressor(n_neighbors=5)
# knn_reg.fit(X_train_s, y_train)
# print(f'R²: {r2_score(y_test, knn_reg.predict(X_test_s)):.4f}')

---
# 6. DECISION TREE 🌳

In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

# ---- ตัวอย่าง ----
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Decision Tree ไม่ต้อง scale

# ---- Train ----
dt = DecisionTreeClassifier(
    max_depth=4,            # ความลึกสูงสุด (None = ไม่จำกัด = overfit)
    min_samples_split=5,    # ขั้นต่ำ samples ก่อน split
    min_samples_leaf=2,     # ขั้นต่ำ samples ใน leaf
    criterion='gini',       # 'gini' หรือ 'entropy'
    random_state=42
)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred))

# ---- Visualize Tree ----
plt.figure(figsize=(20, 8))
plot_tree(
    dt,
    feature_names=data.feature_names,
    class_names=data.target_names,
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3   # แสดงแค่ 3 ระดับ (ไม่ให้รูปใหญ่เกิน)
)
plt.title('Decision Tree')
plt.show()

# ---- Feature Importance ----
fi = pd.Series(dt.feature_importances_, index=data.feature_names)
fi.sort_values(ascending=True).tail(10).plot(
    kind='barh', figsize=(8,5), title='Feature Importance'
)
plt.tight_layout()
plt.show()

# ---- หา max_depth ที่ดีที่สุด ----
depths = range(1, 15)
test_accs = []
for d in depths:
    dt_d = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt_d.fit(X_train, y_train)
    test_accs.append(dt_d.score(X_test, y_test))

best_d = depths[np.argmax(test_accs)]
print(f'Best max_depth: {best_d}')
plt.figure(figsize=(7,4))
plt.plot(depths, test_accs, 'g-o')
plt.xlabel('max_depth'); plt.ylabel('Test Accuracy')
plt.title('max_depth Tuning')
plt.grid(True)
plt.show()

---
# 7. RANDOM FOREST 🌲🌲🌲

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# ---- ตัวอย่าง ----
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---- Train ----
rf = RandomForestClassifier(
    n_estimators=100,     # จำนวนต้นไม้ (มากขึ้น = ดีขึ้น แต่ช้าขึ้น)
    max_depth=None,       # None = ปล่อยเต็มที่
    max_features='sqrt',  # จำนวน features ต่อ split ('sqrt','log2',int)
    min_samples_leaf=1,
    n_jobs=-1,            # ใช้ทุก CPU core
    random_state=42
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred))

# ---- Feature Importance ----
fi = pd.Series(rf.feature_importances_, index=data.feature_names)
fi.sort_values(ascending=True).tail(12).plot(
    kind='barh', figsize=(9,6), title='Random Forest Feature Importance'
)
plt.tight_layout()
plt.show()

# ---- Cross Validation ----
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
print(f'CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# ---- Regression version ----
# rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
# rf_reg.fit(X_train, y_train)
# print(f'R²: {r2_score(y_test, rf_reg.predict(X_test)):.4f}')

---
# 8. ENSEMBLE METHODS ⚡

---
# 8.5 STACKING 🏗️

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.datasets import load_breast_cancer
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---- Base learners (Level-0) ----
base_learners = [
    ('knn',  KNeighborsClassifier(n_neighbors=5)),
    ('dt',   DecisionTreeClassifier(max_depth=4, random_state=42)),
    ('svc',  SVC(probability=True, random_state=42)),  # probability=True สำหรับ soft voting
]

# ---- Meta learner (Level-1) ----
meta_learner = LogisticRegression(max_iter=1000)

# ---- Stacking ----
stack = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5,              # ใช้ 5-fold สร้าง meta-features
    stack_method='auto',
    passthrough=False  # True = ส่ง X ต้นฉบับไปให้ meta learner ด้วย
)

from sklearn.pipeline import make_pipeline
stack_pipe = make_pipeline(StandardScaler(), stack)
stack_pipe.fit(X_train, y_train)
y_pred = stack_pipe.predict(X_test)

print(f'Stacking Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    BaggingClassifier,
    VotingClassifier,
)

from sklearn.datasets import load_breast_cancer
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ============================================================
# AdaBoost
# ============================================================
ada = AdaBoostClassifier(
    n_estimators=100,     # จำนวน weak learners
    learning_rate=0.1,    # ขนาด step (เล็กลง = ดีขึ้น แต่ต้องการ estimators มากขึ้น)
    random_state=42
)
ada.fit(X_train, y_train)
print(f'AdaBoost Accuracy:  {accuracy_score(y_test, ada.predict(X_test)):.4f}')

# ============================================================
# Gradient Boosting
# ============================================================
gbm = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,           # ลึกน้อยๆ ป้องกัน overfit
    subsample=0.8,         # random sample 80% ต่อ tree
    random_state=42
)
gbm.fit(X_train, y_train)
print(f'GradBoost Accuracy: {accuracy_score(y_test, gbm.predict(X_test)):.4f}')

# ============================================================
# XGBoost (ถ้าติดตั้ง)
# ============================================================
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
    xgb.fit(X_train, y_train)
    print(f'XGBoost Accuracy:   {accuracy_score(y_test, xgb.predict(X_test)):.4f}')
except ImportError:
    print('XGBoost not installed, use: pip install xgboost')

# ============================================================
# Extra Trees
# ============================================================
et = ExtraTreesClassifier(n_estimators=100, random_state=42)
et.fit(X_train, y_train)
print(f'Extra Trees Acc:    {accuracy_score(y_test, et.predict(X_test)):.4f}')

# ============================================================
# Bagging  – train base estimators บน bootstrap samples
# ============================================================
from sklearn.tree import DecisionTreeClassifier
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),   # base estimator (default = Decision Tree)
    n_estimators=100,     # จำนวน base models
    max_samples=0.8,      # % ของ rows ที่สุ่มต่อ model (bootstrap)
    max_features=0.8,     # % ของ columns ที่สุ่มต่อ model
    bootstrap=True,       # True = สุ่มแบบคืน (bagging), False = สุ่มแบบไม่คืน (pasting)
    n_jobs=-1,
    random_state=42
)
bag.fit(X_train, y_train)
print(f'Bagging Accuracy:   {accuracy_score(y_test, bag.predict(X_test)):.4f}')
# หมายเหตุ: Random Forest คือ Bagging + random feature selection
# ถ้าอยากได้ feature importance ใช้ Random Forest ดีกว่า

# ============================================================
# Voting Classifier – รวมหลาย model
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

voting = VotingClassifier(
    estimators=[
        ('rf',  RandomForestClassifier(n_estimators=50, random_state=42)),
        ('gbm', GradientBoostingClassifier(n_estimators=50, random_state=42)),
        ('lr',  LogisticRegression(max_iter=500)),
    ],
    voting='soft'   # 'soft'=ใช้ probability, 'hard'=ใช้ majority vote
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
voting_pipeline = Pipeline([('sc', StandardScaler()), ('clf', voting)])
voting_pipeline.fit(X_train, y_train)
print(f'Voting Accuracy:    {accuracy_score(y_test, voting_pipeline.predict(X_test)):.4f}')

---
# 9. EVALUATION SUMMARY 📊

In [ ]:
# ============================================================
# Phase 5 – EVALUATION: CLASSIFICATION 🎯
# ============================================================
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s  = sc.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_s, y_train)
y_pred  = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]

# ---- Scalar Metrics ----
print("=" * 48)
print(f"  Accuracy : {accuracy_score(y_test, y_pred):.4f}  (ถูกทั้งหมด / ทั้งหมด)")
print(f"  Precision: {precision_score(y_test, y_pred):.4f}  (ที่ทำนาย P จริง P เท่าไหร่)")
print(f"  Recall   : {recall_score(y_test, y_pred):.4f}  (จาก P ทั้งหมด จับได้เท่าไหร่)")
print(f"  F1-Score : {f1_score(y_test, y_pred):.4f}  (Harmonic mean ของ P & R)")
print(f"  AUC-ROC  : {roc_auc_score(y_test, y_proba):.4f}  (พื้นที่ใต้ ROC curve)")
print("=" * 48)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

# ---- Plot: Confusion Matrix + ROC + PR Curve ----
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=data.target_names, yticklabels=data.target_names)
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_val = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, lw=2, label=f'AUC = {auc_val:.3f}')
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Precision-Recall Curve
precision_c, recall_c, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
axes[2].plot(recall_c, precision_c, lw=2, label=f'AP = {ap:.3f}')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# ---- Confusion Matrix ทำความเข้าใจ ----
tn, fp, fn, tp = cm.ravel()
print(f"\nTP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"Sensitivity (Recall)  = TP/(TP+FN) = {tp/(tp+fn):.4f}  (จับ positive ได้กี่%)")
print(f"Specificity           = TN/(TN+FP) = {tn/(tn+fp):.4f}  (จับ negative ได้กี่%)")
print(f"Positive Pred Value (Precision) = TP/(TP+FP) = {tp/(tp+fp):.4f}")


In [ ]:
# ============================================================
# Phase 5 – EVALUATION: REGRESSION 📉
# ============================================================
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error
)

data_r = load_diabetes()
X_r, y_r = data_r.data, data_r.target
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)
sc_r = StandardScaler()
X_tr_s = sc_r.fit_transform(X_tr)
X_te_s  = sc_r.transform(X_te)

model_r = LinearRegression()
model_r.fit(X_tr_s, y_tr)
y_pred_r = model_r.predict(X_te_s)

mae  = mean_absolute_error(y_te, y_pred_r)
mse  = mean_squared_error(y_te, y_pred_r)
rmse = np.sqrt(mse)
r2   = r2_score(y_te, y_pred_r)
mape = mean_absolute_percentage_error(y_te, y_pred_r) * 100

print("=" * 48)
print(f"  MAE   : {mae:.2f}   (error เฉลี่ย, unit เดิม)")
print(f"  MSE   : {mse:.2f}  (penalize outlier มากกว่า MAE)")
print(f"  RMSE  : {rmse:.2f}   (MAE แบบ root, unit เดิม)")
print(f"  MAPE  : {mape:.2f}%   (% error เฉลี่ย)")
print(f"  R²    : {r2:.4f}   (1.0=perfect, 0=predict mean, <0=worse)")
print("=" * 48)
print("  ดี/แย่: R² ใกล้ 1 = ดี | RMSE ยิ่งต่ำ = ดี")

# ---- Plot: Actual vs Predicted + Residuals ----
residuals = y_te - y_pred_r
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Actual vs Predicted
axes[0].scatter(y_te, y_pred_r, alpha=0.6, color='steelblue', s=30)
lims = [min(y_te.min(), y_pred_r.min()), max(y_te.max(), y_pred_r.max())]
axes[0].plot(lims, lims, 'r--', lw=2)
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Actual vs Predicted  (R²={r2:.3f})')
axes[0].grid(True, alpha=0.3)

# Residual Plot
axes[1].scatter(y_pred_r, residuals, alpha=0.6, color='orange', s=30)
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residuals (Actual - Pred)')
axes[1].set_title('Residual Plot  (กระจายรอบ 0 = ดี)')
axes[1].grid(True, alpha=0.3)

# Residual Distribution
axes[2].hist(residuals, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
axes[2].axvline(0, color='red', linestyle='--', lw=2)
axes[2].set_xlabel('Residual'); axes[2].set_ylabel('Count')
axes[2].set_title('Residual Distribution  (Normal = ดี)')

plt.tight_layout(); plt.show()


---
# 9.5 CROSS VALIDATION 🔁

In [ ]:
from sklearn.model_selection import (
    cross_val_score, cross_validate,
    KFold, StratifiedKFold, LeaveOneOut,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)

# ============================================================
# 1. cross_val_score – เร็วที่สุด
# ============================================================
model = RandomForestClassifier(n_estimators=100, random_state=42)

# K-Fold ธรรมดา (classification ควรใช้ Stratified)
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f'KFold CV:          mean={cv_scores.mean():.4f}  std={cv_scores.std():.4f}')

# Stratified K-Fold (รักษา class ratio ในแต่ละ fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_s = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
print(f'Stratified KFold:  mean={cv_scores_s.mean():.4f}  std={cv_scores_s.std():.4f}')

# ============================================================
# 2. cross_validate – ดู metric หลายตัวพร้อมกัน
# ============================================================
cv_result = cross_validate(
    model, X, y, cv=skf,
    scoring=['accuracy', 'f1', 'roc_auc'],
    return_train_score=True
)
print(f"\nTest  Accuracy: {cv_result['test_accuracy'].mean():.4f}")
print(f"Test  F1:       {cv_result['test_f1'].mean():.4f}")
print(f"Test  AUC-ROC:  {cv_result['test_roc_auc'].mean():.4f}")
print(f"Train Accuracy: {cv_result['train_accuracy'].mean():.4f}  (ถ้าสูงกว่า test มาก = overfit)")

# ============================================================
# 3. GridSearchCV – หา hyperparameter ที่ดีที่สุด
# ============================================================
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 5, 10],
    'max_features': ['sqrt', 'log2'],
}
gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
gs.fit(X, y)
print(f'\nGridSearch Best Params: {gs.best_params_}')
print(f'GridSearch Best Score:  {gs.best_score_:.4f}')

# ============================================================
# 4. RandomizedSearchCV – เร็วกว่า Grid เมื่อ param เยอะ
# ============================================================
from scipy.stats import randint
param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth':    [None, 3, 5, 10, 20],
    'max_features': ['sqrt', 'log2'],
}
rs = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist,
    n_iter=20,      # สุ่มแค่ 20 ชุด param
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)
rs.fit(X, y)
print(f'\nRandomSearch Best Params: {rs.best_params_}')
print(f'RandomSearch Best Score:  {rs.best_score_:.4f}')

In [ ]:
# ======================================================
# เปรียบทุก model ในคราวเดียว
# ======================================================
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

X, y = load_breast_cancer(return_X_y=True)

models = {
    'Logistic Reg':  LogisticRegression(max_iter=1000),
    'KNN':           KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boost':GradientBoostingClassifier(n_estimators=100, random_state=42),
    'AdaBoost':      AdaBoostClassifier(n_estimators=100, random_state=42),
}

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

results = {}
for name, model in models.items():
    pipe = make_pipeline(StandardScaler(), model)
    scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
    results[name] = scores

results_df = pd.DataFrame(results)
print('5-Fold CV Accuracy:')
print(results_df.mean().sort_values(ascending=False).round(4))

results_df.boxplot(figsize=(11, 5))
plt.xticks(rotation=15)
plt.title('Model Comparison (5-Fold CV Accuracy)')
plt.ylabel('Accuracy')
plt.tight_layout()
plt.show()

---
# 10. DATA PREP TEMPLATE 🧹
**คัดลอกไปใช้ได้เลยเมื่อรับโจทย์**

---
# 10.5 ENCODING CATEGORICAL VARIABLES 🏷️

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

# ---- Mock data ----
df = pd.DataFrame({
    'color':    ['red', 'blue', 'green', 'blue', 'red'],   # nominal – no order
    'size':     ['S', 'M', 'L', 'XL', 'M'],                # ordinal – has order
    'gender':   ['M', 'F', 'M', 'F', 'M'],                 # binary
    'label':    ['cat', 'dog', 'cat', 'bird', 'dog'],       # target
})
print(df)
print()

# ============================================================
# 1. LabelEncoder  ← ใช้กับ target (y) เท่านั้น
#    แปลง text → int  (alphabetical order)
# ============================================================
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])
# bird=0, cat=1, dog=2
print("LabelEncoder (target):")
print(df[['label', 'label_enc']])
print("Classes:", le.classes_)          # decode กลับด้วย le.inverse_transform([0,1,2])
print()

# ============================================================
# 2. map()  ← ใช้กับ binary feature (2 ค่า) เพราะกำหนดเองได้
# ============================================================
df['gender_enc'] = df['gender'].map({'M': 0, 'F': 1})
print("map() binary feature:")
print(df[['gender', 'gender_enc']])
print()

# ============================================================
# 3. OrdinalEncoder  ← ใช้กับ ordinal feature (มี order)
#    ต้องกำหนด categories เองเพื่อควบคุม order
# ============================================================
oe = OrdinalEncoder(categories=[['S', 'M', 'L', 'XL']])
df['size_enc'] = oe.fit_transform(df[['size']]).astype(int)
# S=0, M=1, L=2, XL=3
print("OrdinalEncoder (ordered):")
print(df[['size', 'size_enc']])
print()

# ============================================================
# 4. OneHotEncoder  ← ใช้กับ nominal feature (ไม่มี order)
#    สร้าง column ใหม่ 1 column ต่อ 1 category
# ============================================================
ohe = OneHotEncoder(sparse_output=False, drop='first')   # drop='first' ลด multicollinearity
color_enc = ohe.fit_transform(df[['color']])
color_cols = ohe.get_feature_names_out(['color'])         # ['color_green', 'color_red']
df_ohe = pd.concat([df, pd.DataFrame(color_enc, columns=color_cols)], axis=1)
print("OneHotEncoder (nominal):")
print(df_ohe[['color'] + list(color_cols)])
print()

# ---- หรือใช้ pd.get_dummies (เร็วกว่าสำหรับใน notebook) ----
df_dummies = pd.get_dummies(df[['color']], drop_first=True)
print("pd.get_dummies (same result, shorter code):")
print(df_dummies)
print()

# ============================================================
# สรุป: เลือกใช้อะไร?
# ============================================================
print("""
WHEN TO USE:
  LabelEncoder   → target (y) เท่านั้น
  map()          → binary feature (M/F, Y/N) – กำหนดค่าเองได้
  OrdinalEncoder → ordinal feature (S<M<L<XL) – ต้องระบุ order
  OneHotEncoder  → nominal feature (สี, ประเภท) – ไม่มี order
  get_dummies()  → เหมือน OneHotEncoder แต่ใช้สั้นกว่าใน pandas
""")

In [ ]:
# ============================================================
# TEMPLATE – วางไว้บนสุดของ notebook สอบ
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings; warnings.filterwarnings('ignore')

# ---- 1. โหลดข้อมูล ----
df = pd.read_csv('YOUR_FILE.csv')          # <-- เปลี่ยน

# ---- 2. EDA เบื้องต้น ----
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df.describe())
df.head()

# ---- 3. จัดการ Missing ----
# df.dropna(inplace=True)
# df['col'].fillna(df['col'].median(), inplace=True)
# df['cat_col'].fillna(df['cat_col'].mode()[0], inplace=True)

# ---- 4. Encode Categorical ----
# df = pd.get_dummies(df, drop_first=True)           # One-Hot
# df['col'] = LabelEncoder().fit_transform(df['col']) # Label

# ---- 5. แยก X, y ----
X = df.drop('TARGET_COL', axis=1)         # <-- เปลี่ยน
y = df['TARGET_COL']                      # <-- เปลี่ยน

# ---- 6. Split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
    # stratify=y   # เพิ่มถ้าเป็น classification
)

# ---- 7. Scale ----
sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s  = sc.transform(X_test)    # transform เท่านั้น! ห้าม fit อีก

# ---- 8. Train & Evaluate ----
# from sklearn.XXX import YYYClassifier
# model = YYYClassifier(...)
# model.fit(X_train_s, y_train)
# y_pred = model.predict(X_test_s)
# print(accuracy_score(y_test, y_pred))
# print(classification_report(y_test, y_pred))

---
# ⚠️ จุดระวังสำคัญในข้อสอบ

| เรื่อง | ต้องทำ |
|--------|--------|
| **KNN** | ต้อง Scale ก่อนเสมอ (ไม่ Scale = ผิดแน่) |
| **Logistic Reg** | ใส่ `max_iter=1000` ถ้ามี ConvergenceWarning |
| **Decision Tree** | ใส่ `max_depth` ไม่อย่างนั้น overfit |
| **Scale** | fit บน train เท่านั้น → `.transform()` บน test |
| **Stratify** | ใส่ `stratify=y` ใน split ถ้าเป็น classification |
| **K-Means** | Scale ก่อน → ดู Elbow → เลือก k → fit |
| **ส่งงาน** | File > Download .ipynb → ส่งไฟล์ ห้ามส่ง link! |